# DR04 â€” Compositional Evaluation & Conditioning Adherence

This notebook is a thin reproducibility/reporting layer for the closed DR04 protocol. It loads frozen metadata only: it does not decode sealed images, generate samples, retrain, or report generated-image results. Image quality is evaluated separately in DR05.

## Scientific question and frozen evaluator

DR04 measures conditioning adherence and compositional generalization. The evaluator was trained from scratch on real images only and is a shared small CNN with four attribute heads (`face_shape`, `glasses`, `hair_color`, `face_color`), 293064 parameters. It is an evaluation instrument, not an oracle.

Canonical checkpoint: epoch 41; SHA256 `3f97b8763dc8caa1148ff9434224cac619507f6a377b32bc742cd5fae204c50f`.

The D31 reliability gate is PASS. Its real-image ceiling is reliability evidence only and never numerically corrects generated scores.

In [1]:
from pathlib import Path
import hashlib, json
from collections import Counter, defaultdict

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
ART = ROOT / 'data' / 'processed' / 'dr04_final_eval_v1'
def load(name):
    return json.loads((ART / name).read_text(encoding='utf-8'))
def filehash(name):
    return hashlib.sha256((ART / name).read_bytes()).hexdigest()
protocol = load('final_evaluation_protocol.json')
panel = load('final_prompt_panel_manifest.json')
controls = load('matched_controls_manifest.json')
seeds = load('seed_manifest.json')
edits = load('minimal_edit_pair_manifest.json')
human = load('human_audit_protocol.json')
freeze = load('dr04_freeze_manifest.json')
provenance = load('final_eval_provenance.json')
diagnostics = load('panel_diagnostics.json')
print(f"DR04 protocol: {freeze['status']}; main prompts: {panel['prompt_count']}")

DR04 protocol: CLOSED; main prompts: 252


## Frozen metrics, aggregation, and uncertainty

Primary conditioning metric: **MAA**. Primary compositional metric: **MacroPSR**. Secondary reports include EAM, per-attribute accuracy/Macro-F1/Balanced Accuracy, per-pair PSR, p11/p10/p01/p00, confusion matrices, matched-control diagnostics, minimal-edit sensitivity, and human audit.

Aggregation is `seed â†’ prompt â†’ held-out pair/domain â†’ macro`. The planned 95% percentile bootstrap uses 10,000 repetitions, seed 2026, prompt-cluster resampling with all five seeds together; ID/OOD gaps use the frozen independent resampling rule. No generated results exist yet, so no bootstrap runs here.

## Canonical final OOD / matched-ID protocol

The full OOD panel supports OOD metrics. Matched comparisons use only OOD prompts with both controls. D40/D41-v1 (200 controls) was structurally infeasible; A15 exact MILP established this and A16 established balanced `K=19`. Thus D40/D41-v2 has 152 controls. The notebook does not rerun these audits; their hashes are retained in the freeze manifest.

In [2]:
records = panel['prompt_records']
ood = [r for r in records if r['domain'] == 'OOD']
ids = [r for r in records if r['domain'] == 'ID']
assert len(records) == 252 == len({r['prompt_id'] for r in records}) == len({r['caption'] for r in records})
assert len(ood) == 100 and len(ids) == 152
summary = []
for pair in ('OOD1', 'OOD2', 'OOD3', 'OOD4'):
    pair_ood = [r for r in ood if r['pair_id'] == pair]
    a = [r for r in ids if r['source_pair_id'] == pair and r['control_type'] == 'A_FIXED']
    b = [r for r in ids if r['source_pair_id'] == pair and r['control_type'] == 'B_FIXED']
    summary.append((pair, len(pair_ood), len(a), len(b)))
summary

[('OOD1', 25, 19, 19),
 ('OOD2', 25, 19, 19),
 ('OOD3', 25, 19, 19),
 ('OOD4', 25, 19, 19)]

## Matching quality

Nuisance cost is the number of mismatches among the two non-held-out attributes. It is a metadata matching diagnostic, not generator performance.

In [3]:
by_prompt = defaultdict(list)
for r in ids: by_prompt[r['matched_ood_prompt_id']].append(r)
coverage = Counter(len(by_prompt[r['prompt_id']]) for r in ood)
costs = Counter(r['nuisance_cost'] for r in ids)
assert coverage == Counter({2: 76, 0: 24})
assert costs == Counter({1: 92, 0: 60})
{'coverage': dict(coverage), 'costs': dict(costs), 'mean_cost': sum(r['nuisance_cost'] for r in ids) / len(ids)}

{'coverage': {2: 76, 0: 24},
 'costs': {0: 60, 1: 92},
 'mean_cost': 0.6052631578947368}

## Seeds and planned generation

Generation seeds are separate in role from prompt-selection, bootstrap, and evaluator-training seeds even where all use 2026. The main plan is **252 Ã— 5 = 1260** slots and remains planned/not executed.

In [4]:
assert seeds['generation_seeds'] == [2026, 2027, 2028, 2029, 2030]
assert seeds['expected_main_generation_count'] == 1260
seeds

{'bootstrap_seed': 2026,
 'evaluator_training_seed': 2026,
 'expected_main_generation_count': 1260,
 'expected_minimal_edit_generation_count': 400,
 'generation_seeds': [2026, 2027, 2028, 2029, 2030],
 'prompt_panel_selection_seed': 2026}

## Same-seed minimal-edit diagnostic

Forty TRAIN-known ordinary minimal-edit pairs use the same seed for opposite endpoints. The secondary diagnostic reports Target Edit Success and non-target retention; it is not causal proof. Expected plan: **40 Ã— 2 Ã— 5 = 400** slots. No images are generated here.

In [5]:
pairs = edits['pair_records']
assert len(pairs) == 40
assert len({c for p in pairs for c in (p['caption_a'], p['caption_b'])}) == 80
assert Counter(p['target_attribute'] for p in pairs) == Counter({'face_shape': 10, 'glasses': 10, 'hair_color': 10, 'face_color': 10})
[(p['target_attribute'], p['pair_id']) for p in pairs[:4]]

[('face_shape', 'ME_face_shape_00'),
 ('face_shape', 'ME_face_shape_01'),
 ('face_shape', 'ME_face_shape_02'),
 ('face_shape', 'ME_face_shape_03')]

## Human audit and invalid-generation policy

The future human audit has 80 images (40 ID, 40 OOD), two blinded raters, adjudication on disagreement, four atomic questions, and up to 16 frozen-rule error-enriched samples. It never recalibrates the evaluator.

D39 retains every finite `[3,32,32]` generation; only technical external failures may retry with the exact prompt/seed. No replacement seed is allowed, and persistent intrinsic failures remain in the denominator.

In [6]:
assert human['image_count'] == 80 and human['breakdown'] == {'ID': 40, 'OOD': 40}
{'human_audit': human['breakdown'], 'protocol_status': freeze['status'], 'generation_executed': freeze['final_generation_executed']}

{'human_audit': {'ID': 40, 'OOD': 40},
 'protocol_status': 'CLOSED',
 'generation_executed': False}

## Provenance and closure summary

Generator SHA256: `0bad000254c59bef3b9cf3a2e33ccc10d263f6cea231841e71950a41b7bd3884`.
Evaluator SHA256: `3f97b8763dc8caa1148ff9434224cac619507f6a377b32bc742cd5fae204c50f`.
D31 freeze SHA256: `374e23ab2cb42f9b21b813042e93cfbfcab97187fdd30621b154378fd7c2f436`.

**DR04 evaluator reliability: PASS. DR04 protocol: FROZEN / CLOSED. Main planned samples: 1260. Minimal-edit planned samples: 400. Total: 1660. Final generator evaluation executed: NO.**

The scientific protocol is closed; final generation/evaluation is deferred until the global final-evaluation protocol permits it.

In [7]:
assert provenance['generator_checkpoint_sha256'].startswith('0bad000254')
assert provenance['evaluator_checkpoint_sha256'].startswith('3f97b8763d')
{name: filehash(name) for name in ['dr04_freeze_manifest.json', 'final_prompt_panel_manifest.json', 'minimal_edit_pair_manifest.json']} 

{'dr04_freeze_manifest.json': 'c849a50ae342803707c20090d8ed5cd1383ceac9854d2e6b035a2519ee6b4af6',
 'final_prompt_panel_manifest.json': '102c76b5281119df066cc83e59ca7dfd836aa99bb3ab511b0628f3d750224e60',
 'minimal_edit_pair_manifest.json': '04b9a83acbca2e386eca3b3702a7ca2880b601debd2d8d3ed30fde57b7c1847e'}